In [ ]:
%%configure  

{ 
    "vCores": 
    { 
        "parameterName": "pipelinecore", 
        "defaultValue": 2 
    }
}

In [ ]:
!pip install -q duckrun --upgrade
notebookutils.session.restartPython()


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
try:
    import notebookutils
    vl                     = notebookutils.variableLibrary.getLibrary("deploy_config")
    workspace_id           = vl.workspace_id
    lakehouse_name         = vl.lakehouse_name
    lakehouse_landing_name = vl.lakehouse_landing_name
    download_limit         = vl.download_limit
    lakehouse_id           = notebookutils.lakehouse.get(lakehouse_name).get('id')
    landing_lakehouse_id   = notebookutils.lakehouse.get(lakehouse_landing_name).get('id')
    token                  = notebookutils.credentials.getToken('storage')
    dbt_target             = 'dev'
    # Mount the TRANSFORM lakehouse only -- it is the one holding Files/dbt. The raw archive
    # in the landing lakehouse is read by absolute abfss path, never through the mount.
    notebookutils.fs.mount(
        f'abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}',
        '/lh',
        {'fileCacheTimeout': 0},
    )
    mount_path = notebookutils.fs.getMountPath('/lh')
    dbt_path   = f'{mount_path}/Files/dbt'
except ModuleNotFoundError:
    from azure.identity import AzureCliCredential
    import duckrun
    workspace_id           = "91588e42-0f1c-4e56-bcaa-cbf015b8f312"
    lakehouse_name         = "data"
    lakehouse_landing_name = "data_landing"
    download_limit         = "2"
    dbt_path               = "/lakehouse/default/Files/dbt_fabric_python_notebook_delta/dbt"
    _ws                    = duckrun.workspace(workspace_id)
    lakehouse_id           = _ws.lakehouse_id(lakehouse_name)
    landing_lakehouse_id   = _ws.lakehouse_id(lakehouse_landing_name)
    token                  = AzureCliCredential().get_token("https://storage.azure.com/.default").token
    dbt_target             = 'dev'
os.environ['download_limit']   = download_limit

In [ ]:
# The two roots point at DIFFERENT lakehouses: Tables at the transform lakehouse dbt builds
# into, Files at the landing lakehouse holding the raw AEMO archive. dbt keeps one root_path,
# so every ref() still resolves inside one lakehouse and no model crosses.
os.environ['FILES_PATH']          = f'abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{landing_lakehouse_id}/Files'
os.environ['ONELAKE_TABLES_PATH'] = f'abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables'
os.environ['ONELAKE_TOKEN']       = token

In [ ]:
from dbt.cli.main import dbtRunner
os.chdir(dbt_path)
dbt = dbtRunner()
import datetime
# One timestamped log folder per run, beside the dbt project (NOT inside it) on the OneLake mount so logs survive a crash.
_log_dir = f"{os.path.dirname(dbt_path)}/dbt_runs_logs/run-{datetime.datetime.now():%Y%m%dT%H%M%S}"
base = ["--target", dbt_target, "--profiles-dir", ".",
        "--log-path", _log_dir, "--target-path", "/tmp/dbt_target"]

# Same flow as CI: ONE run. Every model is a table, so the DAG order is all the sequencing
# there is -- no check_new_daily probe, no --full-refresh branch, no per-model excludes.
# stg_csv_archive_log downloads new CSVs as a side effect and everything downstream rebuilds
# from the whole archive.
result = dbt.invoke(["run", *base])
if not result.success:
    print("dbt run had failures -- retrying failed models once...")
    _ = dbt.invoke(["retry", *base])

_ = dbt.invoke(["test", *base])

In [ ]:
# Refresh the ontology graph so the Operations Agent sees the data this run just landed.
# Only SCHEMA changes auto-refresh; data changes need this explicit job. Inlined rather
# than imported: only the dbt project lives on the lakehouse, not the repo. Tolerant: no
# graph deployed, or a refresh already in flight (Fabric rejects the second POST), must
# never fail the load.
import requests
from duckrun.auth import get_fabric_token

_api = "https://api.fabric.microsoft.com/v1"
try:
    _h = {"Authorization": f"Bearer {get_fabric_token()}"}
    _graph = next((m for m in requests.get(f"{_api}/workspaces/{workspace_id}/GraphModels",
                                           headers=_h).json().get("value", [])
                   if m["displayName"].startswith("aemo_nem_graph")), None)
    if _graph is None:
        print("no aemo_nem graph model deployed -- skipping refresh")
    else:
        _r = requests.post(f"{_api}/workspaces/{workspace_id}/items/{_graph['id']}"
                           "/jobs/instances?jobType=RefreshGraph", headers=_h)
        print(f"RefreshGraph on {_graph['displayName']} -> {_r.status_code}")
except Exception as _e:
    print(f"graph refresh skipped: {_e}")